In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableBranch
from langchain.schema.output_parser import StrOutputParser

In [2]:
llm = GoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
)

In [3]:
# Define prompt templates for different feedback types
positive_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "Generate one warm and appreciative thank you note (2-3 lines) for this positive feedback: {feedback}. Make the customer feel valued and appreciated."),
    ]
)

negative_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "Generate one empathetic and solution-focused response (2-3 lines) addressing this negative feedback: {feedback}. Show understanding and offer help."),
    ]
)

neutral_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "Generate one engaging request for more details (2-3 lines) about this neutral feedback: {feedback}. Show interest in their experience."),
    ]
)

escalate_feedback_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "Generate one professional escalation message (2-3 lines) for this feedback: {feedback}. Assure the customer that their concern will be addressed properly."),
    ]
)

# Define the feedback classification template
classification_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "Classify the sentiment of this feedback as positive, negative, neutral, or escalate: {feedback}."),
    ]
)

In [4]:
branches = RunnableBranch(
    (
        lambda x: 'positive' in x.lower(),
        positive_feedback_template | llm | StrOutputParser()
    ),
    (
        lambda x: 'negative' in x.lower(),
        negative_feedback_template | llm | StrOutputParser()
    ),
    (
        lambda x: 'neutral' in x.lower(),
        neutral_feedback_template | llm | StrOutputParser()
    ),
    escalate_feedback_template | llm | StrOutputParser()
)

In [5]:
classification_chain = classification_template | llm | StrOutputParser()

chain = classification_chain | branches

In [ ]:
reviews = [
    # Positive review
    "The product is excellent. I really enjoyed using it and found it very helpful.",
    
    # Negative review  
    "The product is terrible. It broke after just one use and the quality is very poor.",
    
    # Neutral review
    "The product is okay. It works as expected but nothing exceptional.",
    
    # Escalate review
    "I'm not sure about the product yet. Can you tell me more about its features and benefits?"
]

for review in reviews:
    result = chain.invoke({"feedback": review})
    print(f"Feedback: {review}\nResponse: {result}\n")

Feedback: The product is excellent. I really enjoyed using it and found it very helpful.
Response: Thank you so much for your positive feedback! We truly appreciate you taking the time to share your experience. Your satisfaction is our top priority, and your kind words truly make our day!

Feedback: The product is terrible. It broke after just one use and the quality is very poor.
Response: I'm sorry to hear you had a negative experience. Could you please tell me more about what went wrong so I can understand and help find a solution?

Feedback: The product is okay. It works as expected but nothing exceptional.
Response: Thanks for sharing your "Neutral" feedback. We'd love to understand what contributed to that experience. Could you tell us a little more about what made it feel neutral, or what might have made it more positive or negative for you?

Feedback: I'm not sure about the product yet. Can you tell me more about its features and benefits?
Response: Thank you for your feedback.